<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/Qwen_3_TTS_By_(HunaarG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install with FlashAttention support
!pip install -U qwen-tts gradio huggingface_hub
!pip install flash-attn --no-build-isolation
!pip install pydub
!apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.5 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=8dede945

In [2]:
# import gradio as gr
# from qwen_tts import Qwen3TTSModel
# import torch, soundfile as sf, tempfile, gc
# import os

# # --- THÊM: Import thư viện xử lý âm thanh ---
# try:
#     from pydub import AudioSegment
# except ImportError:
#     print("⚠️ Chưa cài pydub. Vui lòng chạy '!pip install pydub' trước!")

# # ================= PERFORMANCE =================
# current_model = None
# current_model_type = None

# torch.backends.cudnn.benchmark = True
# torch.backends.cudnn.conv.fp32_precision = "tf32"
# torch.backends.cuda.matmul.fp32_precision = "tf32"

# print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")

# # ================= MODEL LOADER =================
# def load_model(t):
#     global current_model, current_model_type
#     if current_model_type == t:
#         return current_model

#     if current_model:
#         del current_model
#         gc.collect()
#         torch.cuda.empty_cache()

#     models = {
#         "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
#         "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
#         "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
#     }

#     current_model = Qwen3TTSModel.from_pretrained(
#         models[t],
#         torch_dtype=torch.float16,
#         device_map="cuda:0",
#         attn_implementation="sdpa"
#     )
#     current_model_type = t
#     return current_model

# # ================= FUNCTIONS (GỐC) =================
# def voice_clone(text, audio, transcript, fast):
#     if not text or not audio: return None
#     m = load_model("base")
#     p = m.create_voice_clone_prompt(
#         ref_audio=audio,
#         ref_text=None if fast else transcript,
#         x_vector_only_mode=fast
#     )
#     with torch.inference_mode():
#         w, sr = m.generate_voice_clone(text=text, voice_clone_prompt=p)
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# def custom_voice(text, voice, inst):
#     if not text: return None
#     m = load_model("custom")
#     with torch.inference_mode():
#         w, sr = m.generate_custom_voice(
#             text=text,
#             speaker=voice,
#             instruct=inst if inst.strip() else None
#         )
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# def voice_design(text, desc):
#     if not text or not desc: return None
#     m = load_model("design")
#     with torch.inference_mode():
#         w, sr = m.generate_voice_design(text=text, instruct=desc)
#     f = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#     sf.write(f.name, w[0], sr)
#     return f.name

# # ================= FUNCTION MỚI: PODCAST MAKER =================
# def generate_podcast(script_text):
#     if not script_text: return None

#     # 1. Tải model Custom (Chỉ tải 1 lần)
#     m = load_model("custom")

#     # 2. Cấu hình nhân vật (Bạn có thể sửa Instruction ở đây)
#     CHAR_CONFIG = {
#         "Eric":   {"voice": "eric",   "inst": "Professional male teacher, deep warm voice, educational tone"},
#         "Serena": {"voice": "serena", "inst": "Friendly female host, energetic, happy tone"},
#         "Ryan":   {"voice": "ryan",   "inst": "British male voice, professional and calm"},
#         "Vivian": {"voice": "vivian", "inst": "Young female voice, fast and fun"}
#     }

#     final_audio = AudioSegment.silent(duration=500) # Bắt đầu với 0.5s im lặng
#     gap = AudioSegment.silent(duration=400) # Nghỉ 0.4s giữa các câu

#     lines = script_text.strip().split('\n')

#     # 3. Duyệt từng dòng kịch bản
#     for line in lines:
#         if ":" in line:
#             # Tách tên và lời thoại (VD: "Eric: Hello world")
#             name, text = line.split(":", 1)
#             name = name.strip()
#             text = text.strip()

#             # Kiểm tra xem tên có trong danh sách không
#             config = CHAR_CONFIG.get(name)

#             if config and text:
#                 print(f"🎙️ Đang tạo giọng cho {name}...")
#                 with torch.inference_mode():
#                     w, sr = m.generate_custom_voice(
#                         text=text,
#                         speaker=config["voice"],
#                         instruct=config["inst"]
#                     )

#                 # Lưu file tạm
#                 temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
#                 sf.write(temp_wav.name, w[0], sr)

#                 # Đọc file bằng Pydub và ghép vào
#                 segment = AudioSegment.from_wav(temp_wav.name)
#                 final_audio += segment + gap

#                 # Xóa file tạm cho nhẹ máy
#                 os.unlink(temp_wav.name)
#             else:
#                 print(f"⚠️ Không tìm thấy cấu hình cho nhân vật: {name} (Hoặc lời thoại trống)")

#     # 4. Xuất file cuối cùng
#     output_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
#     final_audio.export(output_path, format="mp3")
#     print("✅ Đã ghép xong Podcast!")
#     return output_path

# # ================= STRONG UI CSS =================
# css = """
# body{
# background:radial-gradient(circle at top,#eef2ff,#e0e7ff,#f8fafc);
# font-family:Inter,system-ui;
# }
# .hero{
# padding:26px;border-radius:20px;
# background:linear-gradient(135deg,#4f46e5,#7c3aed);
# color:white;text-align:center;
# box-shadow:0 20px 50px rgba(0,0,0,.25);
# margin-bottom:25px;
# }
# .hero h1{font-size:28px;margin-bottom:6px}
# .hero p{opacity:.9}
# .socials{display:flex;gap:14px;justify-content:center;flex-wrap:wrap;margin:22px 0}
# .btn{
# padding:12px 22px;border-radius:999px;
# font-weight:600;text-decoration:none;
# color:white;display:flex;gap:8px;
# align-items:center;transition:.35s;
# box-shadow:0 10px 30px rgba(0,0,0,.2)
# }
# .btn:hover{transform:translateY(-4px) scale(1.04)}
# .yt{background:#ff0000}
# .ig{background:linear-gradient(45deg,#f58529,#dd2a7b,#8134af)}
# .wa{background:#25D366}
# .panel{
# background:rgba(255,255,255,.7);
# backdrop-filter:blur(14px);
# border-radius:20px;
# padding:20px;
# box-shadow:0 15px 40px rgba(0,0,0,.15)
# }
# .footer{
# margin-top:30px;text-align:center;
# font-size:14px;color:#444
# }
# """

# # ================= UI =================
# with gr.Blocks(title="Qwen3-TTS | Hunaar Ansari", css=css) as demo:

#     gr.HTML("""
#     <div class="hero">
#         <h1>🎙️ Qwen3-TTS Voice Studio</h1>
#         <p>Professional AI Voice Cloning • Custom Voices • Voice Design</p>
#         <p><b>Created by Hunaar Ansari (Modified for Podcast)</b></p>
#     </div>
#     """)

#     gr.HTML("""
#     <div class="socials">
#         <a class="btn yt" href="http://www.youtube.com/@Hunaarg" target="_blank">YouTube</a>
#         <a class="btn ig" href="https://www.instagram.com/hunaar_ansari/" target="_blank">Instagram</a>
#         <a class="btn wa" href="https://whatsapp.com/channel/0029VabTEup4dTnKxHsYP01W" target="_blank">
#             WhatsApp Channel
#         </a>
#     </div>
#     """)

#     with gr.Tab("🎤 Voice Clone"):
#         with gr.Group(elem_classes="panel"):
#             t = gr.Textbox(lines=4, label="Text")
#             a = gr.Audio(type="filepath", label="Reference Audio")
#             tr = gr.Textbox(lines=2, label="Transcript (optional)")
#             f = gr.Checkbox(value=True, label="Fast Mode")
#             o = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 voice_clone,[t,a,tr,f],o)

#     with gr.Tab("🎭 Custom Voice"):
#         with gr.Group(elem_classes="panel"):
#             t2 = gr.Textbox(lines=4)
#             v = gr.Dropdown(
#                 ["serena","vivian","ono_anna","sohee","aiden","dylan","eric","ryan","uncle_fu"],
#                 value="serena"
#             )
#             i = gr.Textbox(lines=2)
#             o2 = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 custom_voice,[t2,v,i],o2)

#     with gr.Tab("🎨 Voice Design"):
#         with gr.Group(elem_classes="panel"):
#             t3 = gr.Textbox(lines=4)
#             d = gr.Textbox(lines=3)
#             o3 = gr.Audio()
#             gr.Button("Generate Voice", variant="primary").click(
#                 voice_design,[t3,d],o3)

#     # --- TAB MỚI: PODCAST MODE ---
#     with gr.Tab("🎬 Podcast Mode (Kịch bản)"):
#         with gr.Group(elem_classes="panel"):
#             gr.Markdown("### Nhập kịch bản theo mẫu: `Tên: Lời thoại`")
#             gr.Markdown("Nhân vật hỗ trợ: **Eric** (Nam), **Serena** (Nữ), **Ryan**, **Vivian**")

#             script_input = gr.Textbox(
#                 lines=10,
#                 label="Kịch bản (Script)",
#                 placeholder="Eric: Hello everyone, welcome to the show.\nSerena: Hi Eric! I am so happy to be here.\nEric: Today we talk about AI."
#             )
#             podcast_output = gr.Audio(label="File Podcast Hoàn Chỉnh")

#             gr.Button("🎬 TẠO PODCAST & GHÉP FILE", variant="primary").click(
#                 generate_podcast, [script_input], podcast_output
#             )

#     gr.HTML("""
#     <div class="footer">
#         🚀 Built by <b>Hunaar Ansari</b><br>
#         Follow • Join • Learn AI Voice Technology
#     </div>
#     """)

# print("🔥 Qwen3-TTS Ultra UI | Hunaar Ansari")
# demo.launch(share=True, debug=True, theme=gr.themes.Soft())


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    


🚀 GPU Detected: Tesla T4


/tmp/ipython-input-1545500420.py:185: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Qwen3-TTS | Hunaar Ansari", css=css) as demo:


🔥 Qwen3-TTS Ultra UI | Hunaar Ansari
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c5ac899170061aa480.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c5ac899170061aa480.gradio.live


In [3]:
# @title 🎧 STUDIO PODCAST ĐẠO DIỄN AI (Bản Final)
import gradio as gr
from qwen_tts import Qwen3TTSModel
import torch, soundfile as sf, tempfile, gc
import os
import re

# --- 1. Tự động cài thư viện cắt ghép ---
try:
    from pydub import AudioSegment
except ImportError:
    print("⏳ Đang cài đặt thư viện xử lý âm thanh...")
    os.system('pip install pydub')
    os.system('apt-get install -y ffmpeg') # Bắt buộc cho pydub
    from pydub import AudioSegment

# ================= CẤU HÌNH HỆ THỐNG =================
torch.backends.cudnn.benchmark = True
current_model = None

def load_custom_model():
    global current_model
    # Nếu model đã có thì dùng luôn, không tải lại
    if current_model is not None:
        return current_model

    print(f"📥 Đang tải Model Qwen Custom (Chỉ tải 1 lần duy nhất)...")
    try:
        current_model = Qwen3TTSModel.from_pretrained(
            "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
            torch_dtype=torch.float16,
            device_map="cuda:0",
            attn_implementation="sdpa"
        )
        print("✅ Model đã sẵn sàng phục vụ!")
        return current_model
    except Exception as e:
        print(f"❌ Lỗi tải model: {e}")
        return None

# ================= HÀM XỬ LÝ PODCAST THÔNG MINH =================
def generate_podcast_final(script_text,
                           nam_voice_id, nam_default_style,
                           nu_voice_id, nu_default_style):

    if not script_text: return None
    model = load_custom_model()

    # Tạo khoảng lặng
    final_audio = AudioSegment.silent(duration=500)
    gap = AudioSegment.silent(duration=1500) # Nghỉ 0.35s giữa các câu

    lines = script_text.strip().split('\n')
    total_lines = len([l for l in lines if ":" in l])
    print(f"🚀 Bắt đầu sản xuất Podcast ({total_lines} câu thoại)...")

    for index, line in enumerate(lines):
        if ":" in line:
            # 1. Phân tích cú pháp: "Nam (Sad): Hello"
            name_part, text = line.split(":", 1)
            name_part = name_part.strip()
            text = text.strip()

            # 2. Tìm cảm xúc ghi đè trong ngoặc (...)
            override_emotion = ""
            match = re.search(r'\((.*?)\)', name_part)
            if match:
                override_emotion = match.group(1) # Lấy chữ "Sad"

            # 3. Xác định nhân vật (Nam hay Nu)
            clean_name = re.sub(r'\(.*?\)', '', name_part).strip().lower() # Xóa ngoặc để lấy tên gốc

            is_nam = clean_name in ["nam", "eric", "ryan", "man", "thay", "teacher"]

            # 4. Trộn "Style Chung" với "Cảm xúc Riêng"
            # Logic: Giữ đặc điểm giọng gốc (Style chung) + Thêm cảm xúc hiện tại
            if is_nam:
                speaker = nam_voice_id
                if override_emotion:
                    # Nếu có ghi chú (Sad): Thêm vào instruction
                    instruction = f"{nam_default_style}. Currently speaking with a {override_emotion} tone."
                else:
                    # Nếu không: Dùng style mặc định
                    instruction = nam_default_style
            else:
                speaker = nu_voice_id
                if override_emotion:
                    instruction = f"{nu_default_style}. Currently speaking with a {override_emotion} tone."
                else:
                    instruction = nu_default_style

            # 5. Tạo giọng
            if text:
                print(f"🎙️ [{index+1}/{total_lines}] {clean_name.upper()} ({override_emotion if override_emotion else 'Mặc định'}): {text[:20]}...")

                with torch.inference_mode():
                    w, sr = model.generate_custom_voice(
                        text=text,
                        speaker=speaker,
                        instruct=instruction # AI sẽ đọc theo chỉ đạo này
                    )

                # Lưu file tạm và ghép
                temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                sf.write(temp_wav.name, w[0], sr)
                segment = AudioSegment.from_wav(temp_wav.name)
                final_audio += segment + gap
                os.unlink(temp_wav.name)

    # Xuất file cuối
    output_path = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    final_audio.export(output_path, format="mp3")
    print("✅ Đã xong! Mời nghe thử bên dưới.")
    return output_path

# ================= GIAO DIỆN CHUYÊN NGHIỆP =================
css = """
body{background: #1e1e2e; color: #cdd6f4;}
.panel{background: #313244; padding: 15px; border-radius: 10px; margin-bottom: 10px;}
.input-script textarea {background: #45475a; color: white; font-family: monospace;}
"""

with gr.Blocks(title="Final Podcast Studio", css=css, theme=gr.themes.Base()) as demo:
    gr.Markdown("# 🎬 Podcast AI Director (Bản Final)")
    gr.Markdown("Hệ thống tạo Podcast 2 người với khả năng kiểm soát cảm xúc chi tiết.")

    with gr.Row():
        # CỘT TRÁI: CẤU HÌNH ĐẠO DIỄN
        with gr.Column(scale=1):
            with gr.Group(elem_classes="panel"):
                gr.Markdown("### 👨 Cấu hình Nam (Host/Teacher)")
                nam_id = gr.Dropdown(["eric", "ryan"], value="eric", label="Giọng Nam")
                nam_style = gr.Textbox(
                    label="Style Chung (Mặc định)",
                    lines=3,
                    value="Professional male teacher, deep warm voice, calm educational tone, slow pace, clear articulation."
                )

            with gr.Group(elem_classes="panel"):
                gr.Markdown("### 👩 Cấu hình Nữ (MC/Student)")
                nu_id = gr.Dropdown(["serena", "vivian"], value="serena", label="Giọng Nữ")
                nu_style = gr.Textbox(
                    label="Style Chung (Mặc định)",
                    lines=3,
                    value="Friendly female host, energetic, happy tone, speaking with a smile, natural conversational style."
                )

        # CỘT PHẢI: KỊCH BẢN & KẾT QUẢ
        with gr.Column(scale=2):
            with gr.Group(elem_classes="panel"):
                gr.Markdown("### 📜 Kịch bản (Cú pháp: `Tên (Cảm xúc): Lời thoại`)")
                gr.Markdown("*Mẹo: Cảm xúc viết trong ngoặc đơn bằng tiếng Anh: (Sad), (Angry), (Laughing), (Whispering)...*")

                script_input = gr.Textbox(
                    lines=15,
                    elem_classes="input-script",
                    label="Nhập nội dung tại đây...",
                    value="""Nam: Hello everyone, welcome to Pure English.
Nu (Excited): Hi Eric! I am SO happy to be here today!
Nam: Today implies a very serious topic.
Nu (Whispering): Oh no... is it difficult?
Nam (Laughing): Haha, no. It is actually very fun!
Nu (Relieved): Phew! You scared me for a second."""
                )

                btn_run = gr.Button("🚀 SẢN XUẤT PODCAST (RENDER)", variant="primary")
                audio_output = gr.Audio(label="🎧 Thành Phẩm")

    btn_run.click(
        generate_podcast_final,
        inputs=[script_input, nam_id, nam_style, nu_id, nu_style],
        outputs=audio_output
    )

print("🔥 Hệ thống đã sẵn sàng! Bấm link bên dưới để mở Studio.")
demo.launch(share=True, debug=True)

/tmp/ipython-input-3853080394.py:123: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Final Podcast Studio", css=css, theme=gr.themes.Base()) as demo:


🔥 Hệ thống đã sẵn sàng! Bấm link bên dưới để mở Studio.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d4c5a70228f75a7102.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📥 Đang tải Model Qwen Custom (Chỉ tải 1 lần duy nhất)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

...

🎙️ [15/17] NAM (Happy): Perfect! "Vibrant". ...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


🎙️ [16/17] NU (Mặc định): I see. So the key is...


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


🎙️ [17/17] NAM (Mặc định): Yes. Be descriptive....
✅ Đã xong! Mời nghe thử bên dưới.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d4c5a70228f75a7102.gradio.live
